In [ ]:
import importlib
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from md_Helpers import (
    ProjectPaths,
    SQLiteRunDatabase,
    cavitation_dataframe,
    thermalization_dataframe,
)

# Reload the analysis helpers without requiring a kernel restart.
import md_Helpers.nbins_tuning as nbins_tuning
importlib.reload(nbins_tuning)

refit_cavitation_nbins = nbins_tuning.refit_cavitation_nbins
plot_phase_nbins_mixtures = nbins_tuning.plot_phase_nbins_mixtures
plot_phase_liquid_density_vs_nbins = (
    nbins_tuning.plot_phase_liquid_density_vs_nbins
)

RUN_ID = "20260915184126"
NBINS = [8, 10, 15, 20, 25]
DENSITY_XLIM = (0.0, 0.8)

paths = ProjectPaths()
database = SQLiteRunDatabase(paths.database)
database.initialize()

state = pd.concat(
    [
        thermalization_dataframe(database, Run_ID=RUN_ID),
        cavitation_dataframe(database, Run_ID=RUN_ID),
    ],
    ignore_index=True,
).drop_duplicates(subset="Run_ID")

if len(state) != 1:
    raise ValueError(
        f"Expected one row for {RUN_ID}, found {len(state)}"
    )
if state.iloc[0]["Phase_Separation_Status"] != "Separated":
    raise ValueError(f"Run {RUN_ID} is not marked phase separated")

# Ordinary phase-mixture model:
# Poisson vapor + Gaussian liquid + the existing interface convolution.
normal_fits = refit_cavitation_nbins(
    state,
    nbins_values=NBINS,
    project_paths=paths,
)

normal_shape_figure, normal_shape_axes = plot_phase_nbins_mixtures(
    normal_fits,
    title=f"Normal phase-mixture fits: {RUN_ID}",
    liquid_label="Gaussian liquid",
    density_xlim=DENSITY_XLIM,
)
normal_mean_figure, normal_mean_axis = (
    plot_phase_liquid_density_vs_nbins(
        normal_fits,
        title="Gaussian-liquid mean density vs voxel resolution",
    )
)
normal_mean_axis.set_xlim(min(NBINS) - 1, max(NBINS) + 1)

display(
    normal_fits[
        [
            "voxel_nbins",
            "rho_liquid",
            "rho_liquid_unc",
            "rho_gas",
            "rho_gas_unc",
            "gas_weight",
            "liquid_weight",
            "interface_weight",
            "AIC",
            "success",
        ]
    ].rename(
        columns={
            "rho_liquid": "Liquid mean density",
            "rho_liquid_unc": "Liquid mean uncertainty",
            "rho_gas": "Vapor mean density",
            "rho_gas_unc": "Vapor mean uncertainty",
        }
    )
)

plt.show()

In [ ]:
# Skewed phase-mixture model:
# Poisson vapor + skew-normal liquid + the same interface convolution.
refit_skewed_phase_nbins = nbins_tuning.refit_skewed_phase_nbins

skew_fits = refit_skewed_phase_nbins(normal_fits)

skew_shape_figure, skew_shape_axes = plot_phase_nbins_mixtures(
    skew_fits,
    title=f"Skew-normal phase-mixture fits: {RUN_ID}",
    liquid_label="Skew-normal liquid",
    density_xlim=DENSITY_XLIM,
)
skew_mean_figure, skew_mean_axis = (
    plot_phase_liquid_density_vs_nbins(
        skew_fits,
        title="Skew-normal liquid mean density vs voxel resolution",
    )
)
skew_mean_axis.set_xlim(min(NBINS) - 1, max(NBINS) + 1)

display(
    skew_fits[
        [
            "voxel_nbins",
            "rho_liquid",
            "rho_liquid_unc",
            "rho_gas",
            "rho_gas_unc",
            "liquid_scale_density",
            "liquid_shape_alpha",
            "liquid_shape_alpha_unc",
            "gas_weight",
            "liquid_weight",
            "interface_weight",
            "delta_AIC_vs_normal",
            "success",
        ]
    ].rename(
        columns={
            "rho_liquid": "Liquid mean density",
            "rho_liquid_unc": "Liquid mean uncertainty",
            "rho_gas": "Vapor mean density",
            "rho_gas_unc": "Vapor mean uncertainty",
            "liquid_scale_density": "Liquid scale",
            "liquid_shape_alpha": "Liquid skewness alpha",
            "liquid_shape_alpha_unc": "Alpha uncertainty",
            "delta_AIC_vs_normal": "Delta AIC (skew - normal)",
        }
    )
)

plt.show()